In [ ]:
import pandas as pd
import numpy as np
import pyodbc
import os
import time
from datetime import datetime

# غیرفعال کردن هشدارهای غیرضروری
import warnings
warnings.filterwarnings('ignore')

# ============================================================================
# بخش 1: توابع کمکی
# ============================================================================

def get_sql_connection():
    """ایجاد اتصال به SQL Server"""
    return pyodbc.connect(
        'DRIVER={SQL Server};'
        'SERVER=MKZ-DSAS\\DSAS;'
        'DATABASE=DSAS;'
        'UID=datadriven;'
        'PWD=5Rdx@4Rfv1362'
    )


def transform_sql_data(df, output_file):
    """
    تغییر استراکچر داده‌های SQL و ذخیره در فایل اکسل
    """
    print("🔄 مرحله 2: تغییر استراکچر داده...")
    
    # استخراج AssetIDهای یکتا
    unique_ids = df['AssetID'].unique().tolist()
    print(f"🔢 تعداد AssetIDهای یکتا: {len(unique_ids)}")
    
    # ساخت دیتافریم خروجی با ستون‌های مورد نظر
    columns = ['TimeStamp'] + [f'AssetID_{uid}' for uid in unique_ids]
    output_df = pd.DataFrame(columns=columns)
    
    # پردازش تکراری تا خالی شدن دیتافریم اصلی
    row_count = 0
    while not df.empty and unique_ids:
        main_id = unique_ids[0]
        main_subset = df[df['AssetID'] == main_id]
        
        if main_subset.empty:
            unique_ids.pop(0)
            continue
        
        # گرفتن اولین ردیف از AssetID اصلی
        main_row = main_subset.iloc[0]
        main_ts = main_row['TimeStamp']
        main_value = main_row['Value']
        used_indices = [main_row.name]
        
        # پیدا کردن نزدیک‌ترین TimeStamp برای سایر AssetIDها
        row_data = {'TimeStamp': main_ts, f'AssetID_{main_id}': main_value}
        
        for other_id in unique_ids[1:]:
            subset = df[df['AssetID'] == other_id].copy()
            if subset.empty:
                row_data[f'AssetID_{other_id}'] = np.nan
                continue
            
            subset['ts_diff'] = np.abs(subset['TimeStamp'] - main_ts)
            close_rows = subset[subset['ts_diff'] <= 1800]
            
            if not close_rows.empty:
                closest_row = close_rows.sort_values('ts_diff').iloc[0]
                row_data[f'AssetID_{other_id}'] = closest_row['Value']
                used_indices.append(closest_row.name)
            else:
                row_data[f'AssetID_{other_id}'] = np.nan
        
        # حذف ردیف‌های استفاده‌شده
        df.drop(index=used_indices, inplace=True)
        
        # اضافه کردن ردیف جدید به خروجی
        output_df = pd.concat([output_df, pd.DataFrame([row_data])], ignore_index=True)
        row_count += 1
        
        # نمایش پیشرفت
        if row_count % 1000 == 0:
            print(f"   پردازش {row_count:,} رکورد...")
    
    print(f"✅ تعداد رکوردهای پردازش شده: {row_count:,}")
    
    # حذف ردیف‌های دارای NaN
    print("🧹 مرحله 3: حذف ردیف‌های دارای مقادیر خالی...")
    asset_columns = [col for col in output_df.columns if col.startswith('AssetID_')]
    before_count = len(output_df)
    output_df = output_df.dropna(subset=asset_columns, how='any')
    after_count = len(output_df)
    print(f"   حذف {before_count - after_count:,} ردیف دارای مقادیر خالی")
    
    # تبدیل TimeStamp به تاریخ و زمان
    print("📅 مرحله 4: تبدیل زمان‌ها...")
    output_df['date'] = pd.to_datetime(output_df['TimeStamp'], unit='s')
    output_df['RecordDate'] = output_df['date'].dt.date
    output_df['RecordTime'] = output_df['date'].dt.time
    
    # اضافه کردن ستون id
    output_df.insert(0, 'id', range(1, len(output_df) + 1))
    
    # حذف ستون TimeStamp
    output_df.drop(columns=['TimeStamp'], inplace=True)
    
    # مرتب‌سازی بر اساس تاریخ
    output_df.sort_values(by='date', inplace=True)
    
    # ذخیره در فایل اکسل
    print("💾 مرحله 5: ذخیره در فایل اکسل...")
    
    try:
        os.makedirs(os.path.dirname(output_file), exist_ok=True)
        output_df.to_excel(output_file, index=False)
        print(f"✅ فایل با موفقیت ذخیره شد: {output_file}")
        print(f"📊 تعداد رکوردهای نهایی: {len(output_df):,}")
        print(f"📋 تعداد ستون‌ها: {len(output_df.columns)}")
        
        # ذخیره نسخه پشتیبان
        backup_dir = os.path.join(os.path.dirname(output_file), 'backup')
        os.makedirs(backup_dir, exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M")
        backup_file = os.path.join(backup_dir, f'{os.path.basename(output_file).replace(".xlsx", "")}_{timestamp}.xlsx')
        output_df.to_excel(backup_file, index=False)
        print(f"✅ نسخه پشتیبان ذخیره شد: {backup_file}")
        
        return output_df
        
    except Exception as e:
        print(f"❌ خطا در ذخیره فایل: {e}")
        return None


# ============================================================================
# بخش 2: تعریف توابع تحلیل (3 تابع - استخراج داده از SQL)
# ============================================================================

def analysis_1_sql_generator(file_path, output_filename):
    """
    استخراج داده از SQL Server - ژنراتور - کد شماره 1
    """
    print(f"\n{'='*60}")
    print(f"🔄 شروع تحلیل شماره 1 (SQL - ژنراتور)")
    print(f"{'='*60}")
    
    query = """
    SELECT TOP (10000000) [ID]
          ,[AssetID]
          ,[UnitID]
          ,[Value]
          ,[RecordTime]
          ,[RecordDate]
          ,[PersonelID]
          ,[OutofRange]
          ,[ValueType]
          ,[MobileID]
          ,[DateTime]
          ,[TimeStamp]
          ,[Job]
          ,[IsDeleted]
          ,[ShiftCode]
          ,[OnTime]
    FROM [DSAS].[PDA].[Periodic_Values]
    WHERE [UnitID]=11 AND
    ([AssetID]=9357 OR [AssetID]=9343 OR [AssetID]=9344 OR [AssetID]=9362 OR 
     [AssetID]=9363 OR [AssetID]=9364 OR [AssetID]=9365 OR [AssetID]=9366 OR 
     [AssetID]=9367 OR [AssetID]=9371 OR [AssetID]=9372 OR [AssetID]=9373)
    """
    
    try:
        print("📥 مرحله 1: خواندن داده از SQL Server...")
        conn = get_sql_connection()
        df = pd.read_sql(query, conn)
        conn.close()
        print(f"✅ داده با موفقیت از SQL خوانده شد. تعداد رکوردها: {len(df):,}")
        
        # تبدیل و ذخیره
        result_df = transform_sql_data(df, output_filename)
        
        if result_df is not None:
            print(f"\n📊 اطلاعات آماری (ژنراتور):")
            print(f"   بازه تاریخ: {result_df['date'].min()} تا {result_df['date'].max()}")
            print(f"   تعداد AssetIDها: {len([col for col in result_df.columns if col.startswith('AssetID_')])}")
            return True
        else:
            return False
            
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_2_sql_lubrication(file_path, output_filename):
    """
    استخراج داده از SQL Server - سیستم روغن‌کاری - کد شماره 2
    """
    print(f"\n{'='*60}")
    print(f"🔄 شروع تحلیل شماره 2 (SQL - سیستم روغن‌کاری)")
    print(f"{'='*60}")
    
    query = """
    SELECT TOP (10000000) [ID]
          ,[AssetID]
          ,[UnitID]
          ,[Value]
          ,[RecordTime]
          ,[RecordDate]
          ,[PersonelID]
          ,[OutofRange]
          ,[ValueType]
          ,[MobileID]
          ,[DateTime]
          ,[TimeStamp]
          ,[Job]
          ,[IsDeleted]
          ,[ShiftCode]
          ,[OnTime]
    FROM [DSAS].[PDA].[Periodic_Values]
    WHERE [UnitID]=11 AND
    ([AssetID]=9357 OR [AssetID]=9343 OR [AssetID]=8341 OR [AssetID]=8342 OR 
     [AssetID]=8343 OR [AssetID]=8344 OR [AssetID]=8346 OR [AssetID]=9286 OR 
     [AssetID]=9287 OR [AssetID]=9375)
    """
    
    try:
        print("📥 مرحله 1: خواندن داده از SQL Server...")
        conn = get_sql_connection()
        df = pd.read_sql(query, conn)
        conn.close()
        print(f"✅ داده با موفقیت از SQL خوانده شد. تعداد رکوردها: {len(df):,}")
        
        # تبدیل و ذخیره
        result_df = transform_sql_data(df, output_filename)
        
        if result_df is not None:
            print(f"\n📊 اطلاعات آماری (روغن‌کاری):")
            print(f"   بازه تاریخ: {result_df['date'].min()} تا {result_df['date'].max()}")
            print(f"   تعداد AssetIDها: {len([col for col in result_df.columns if col.startswith('AssetID_')])}")
            return True
        else:
            return False
            
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


def analysis_3_sql_generator_duplicate(file_path, output_filename):
    """
    استخراج داده از SQL Server - ژنراتور (تکراری) - کد شماره 3
    """
    print(f"\n{'='*60}")
    print(f"🔄 شروع تحلیل شماره 3 (SQL - ژنراتور - تکراری)")
    print(f"{'='*60}")
    
    query = """
    SELECT TOP (10000000) [ID]
          ,[AssetID]
          ,[UnitID]
          ,[Value]
          ,[RecordTime]
          ,[RecordDate]
          ,[PersonelID]
          ,[OutofRange]
          ,[ValueType]
          ,[MobileID]
          ,[DateTime]
          ,[TimeStamp]
          ,[Job]
          ,[IsDeleted]
          ,[ShiftCode]
          ,[OnTime]
    FROM [DSAS].[PDA].[Periodic_Values]
    WHERE [UnitID]=11 AND
    ([AssetID]=9357 OR [AssetID]=9343 OR [AssetID]=9344 OR [AssetID]=9362 OR 
     [AssetID]=9363 OR [AssetID]=9364 OR [AssetID]=9365 OR [AssetID]=9366 OR 
     [AssetID]=9367 OR [AssetID]=9371 OR [AssetID]=9372 OR [AssetID]=9373)
    """
    
    try:
        print("📥 مرحله 1: خواندن داده از SQL Server...")
        conn = get_sql_connection()
        df = pd.read_sql(query, conn)
        conn.close()
        print(f"✅ داده با موفقیت از SQL خوانده شد. تعداد رکوردها: {len(df):,}")
        
        # تبدیل و ذخیره
        result_df = transform_sql_data(df, output_filename)
        
        if result_df is not None:
            print(f"\n📊 اطلاعات آماری (ژنراتور - تکراری):")
            print(f"   بازه تاریخ: {result_df['date'].min()} تا {result_df['date'].max()}")
            print(f"   تعداد AssetIDها: {len([col for col in result_df.columns if col.startswith('AssetID_')])}")
            return True
        else:
            return False
            
    except Exception as e:
        print(f"❌ خطا: {e}")
        return False


# ============================================================================
# بخش 3: تعریف وظایف (Jobs) - 3 وظیفه
# ============================================================================

def get_analysis_jobs():
    """
    تعریف ۳ وظیفه استخراج داده از SQL
    """
    output_dir = r'second_stage_inputs\G11'
    
    jobs = [
        {
            'name': 'Analysis 1 - SQL Generator',
            'function': analysis_1_sql_generator,
            'file_path': None,  # ورودی از SQL خوانده می‌شود
            'output_filename': os.path.join(output_dir, 'dsas_g11_generator_bearings_output.xlsx')
        },
        {
            'name': 'Analysis 2 - SQL Lubrication',
            'function': analysis_2_sql_lubrication,
            'file_path': None,
            'output_filename': os.path.join(output_dir, 'dsas_g11_lubrication_system_output.xlsx')
        },
        {
            'name': 'Analysis 3 - SQL Generator (Duplicate)',
            'function': analysis_3_sql_generator_duplicate,
            'file_path': None,
            'output_filename': os.path.join(output_dir, 'dsas_g11_generator_bearings_output.xlsx')
        }
    ]
    return jobs


def run_all_analyses():
    """
    اجرای تمام ۳ تحلیل به ترتیب
    """
    print("\n" + "="*80)
    print(f"🚀 شروع اجرای همه تحلیل‌های استخراج SQL (۳ وظیفه)")
    print(f"📅 زمان: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    print("📋 لیست تحلیلها:")
    print("   1. استخراج داده ژنراتور از SQL")
    print("   2. استخراج داده سیستم روغن‌کاری از SQL")
    print("   3. استخراج داده ژنراتور از SQL (تکراری)")
    print("="*80)
    
    jobs = get_analysis_jobs()
    results = []
    
    for i, job in enumerate(jobs, 1):
        print(f"\n{'#'*80}")
        print(f"# اجرای وظیفه {i} از {len(jobs)}: {job['name']}")
        print(f"{'#'*80}")
        
        try:
            # تابع با دو آرگومان (file_path و output_filename) صدا زده می‌شود
            # اما file_path استفاده نمی‌شود چون داده از SQL خوانده می‌شود
            success = job['function'](job['file_path'], job['output_filename'])
            results.append({
                'job_name': job['name'],
                'success': success,
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S')
            })
            
            if success:
                print(f"✅ وظیفه {i} با موفقیت کامل شد")
            else:
                print(f"❌ وظیفه {i} با شکست مواجه شد")
                
        except Exception as e:
            print(f"❌ خطای غیرمنتظره در وظیفه {i}: {e}")
            results.append({
                'job_name': job['name'],
                'success': False,
                'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
                'error': str(e)
            })
    
    # گزارش نهایی
    print("\n" + "="*80)
    print("📊 گزارش نهایی اجرای همه تحلیل‌ها")
    print("="*80)
    
    success_count = sum(1 for r in results if r['success'])
    total_count = len(results)
    
    print(f"✅ موفق: {success_count} از {total_count}")
    print(f"❌ ناموفق: {total_count - success_count} از {total_count}")
    
    for r in results:
        status = "✅" if r['success'] else "❌"
        print(f"   {status} {r['job_name']} - {r['timestamp']}")
    
    print("="*80)
    print(f"🏁 پایان اجرا در {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    
    return results


# ============================================================================
# بخش 4: زمان‌بندی (Scheduler) با دو زمان ۹:۰۰ و ۲۱:۰۰
# ============================================================================

def run_scheduler():
    """
    بررسی مداوم برای اجرا در زمان‌های مشخص (هر روز ساعت ۹:۰۰ و ۲۱:۰۰)
    """
    print("="*80)
    print("🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - استخراج داده از SQL")
    print("="*80)
    print("📋 شامل ۳ وظیفه استخراج داده:")
    print("   1. ژنراتور (Generator)")
    print("   2. سیستم روغن‌کاری (Lubrication)")
    print("   3. ژنراتور (تکراری)")
    print("="*80)
    print("⏰ زمان‌های اجرا (هر روز):")
    print("   - ساعت 09:00")
    print("   - ساعت 21:00")
    print("="*80)
    print("💡 برای توقف برنامه، Ctrl+C را بزنید")
    print("="*80)
    
    last_run_times = {}  # ذخیره زمان‌های اجرا شده
    
    while True:
        try:
            now = datetime.now()
            current_time = now.strftime("%H:%M")
            
            # بررسی زمان‌های مشخص (۹:۰۰ و ۲۱:۰۰)
            if current_time in ["13:55", "21:00"]:
                # فقط چک می‌کنیم که در همین زمان دوبار اجرا نشود
                if last_run_times.get(current_time) != now.strftime("%Y-%m-%d"):
                    print("\n" + "="*80)
                    print(f"⏰ زمان اجرا فرا رسید: {now.strftime('%Y-%m-%d %H:%M:%S')}")
                    print("="*80)
                    
                    # اجرای همه تحلیل‌ها
                    results = run_all_analyses()
                    
                    # ثبت زمان اجرا
                    last_run_times[current_time] = now.strftime("%Y-%m-%d")
                    
                    print("\n" + "="*80)
                    print("✅ اجرای زمان‌بندی شده با موفقیت کامل شد!")
                    print("="*80)
                    
                    # ۶۰ ثانیه صبر کن تا از اجرای مجدد در همان دقیقه جلوگیری شود
                    time.sleep(60)
            
            # هر ۳۰ ثانیه یکبار بررسی کن
            time.sleep(30)
            
        except KeyboardInterrupt:
            print("\n" + "="*80)
            print("⏹️ برنامه با دستور کاربر متوقف شد")
            print(f"⏹️ زمان توقف: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print("="*80)
            break
            
        except Exception as e:
            print(f"❌ خطا در حلقه اصلی: {e}")
            print("🔄 ادامه اجرا...")
            time.sleep(60)


# ============================================================================
# بخش 5: اجرای اصلی
# ============================================================================

if __name__ == "__main__":
    try:
        print("="*80)
        print("🚀 شروع برنامه جامع استخراج داده از SQL Server (۳ وظیفه یکپارچه)")
        print("="*80)
        print("📋 لیست وظایف:")
        print("   1. استخراج داده ژنراتور از SQL → dsas_g11_generator_bearings_output.xlsx")
        print("   2. استخراج داده سیستم روغن‌کاری از SQL → dsas_g11_lubrication_system_output.xlsx")
        print("   3. استخراج داده ژنراتور از SQL (تکراری) → dsas_g11_generator_bearings_output.xlsx")
        print("="*80)
        print("⏰ زمان‌بندی: هر روز ساعت 09:00 و 21:00")
        print("="*80)
        
        # اجرای زمان‌بندی
        run_scheduler()
        
    except Exception as e:
        print(f"❌ خطای غیرمنتظره: {e}")
        import traceback
        traceback.print_exc()
        input("برای خروج Enter بزنید...")

🚀 شروع برنامه جامع استخراج داده از SQL Server (۳ وظیفه یکپارچه)
📋 لیست وظایف:
   1. استخراج داده ژنراتور از SQL → dsas_g11_generator_bearings_output.xlsx
   2. استخراج داده سیستم روغن‌کاری از SQL → dsas_g11_lubrication_system_output.xlsx
   3. استخراج داده ژنراتور از SQL (تکراری) → dsas_g11_generator_bearings_output.xlsx
⏰ زمان‌بندی: هر روز ساعت 09:00 و 21:00
🔄 برنامه زمان‌بندی خودکار شروع به کار کرد - استخراج داده از SQL
📋 شامل ۳ وظیفه استخراج داده:
   1. ژنراتور (Generator)
   2. سیستم روغن‌کاری (Lubrication)
   3. ژنراتور (تکراری)
⏰ زمان‌های اجرا (هر روز):
   - ساعت 09:00
   - ساعت 21:00
💡 برای توقف برنامه، Ctrl+C را بزنید

⏰ زمان اجرا فرا رسید: 2026-07-10 13:55:28

🚀 شروع اجرای همه تحلیل‌های استخراج SQL (۳ وظیفه)
📅 زمان: 2026-07-10 13:55:28
📋 لیست تحلیلها:
   1. استخراج داده ژنراتور از SQL
   2. استخراج داده سیستم روغن‌کاری از SQL
   3. استخراج داده ژنراتور از SQL (تکراری)

################################################################################
# اجرای وظیفه 1 از 3: Analys